### Augmented Data Preparation

This code will prepare the augmented data for evaluation purposes. This will take the random images from the collected data, augment them, and then put same set of open-ended questions for all the augmentations. The purpose of this evaluation is to ensure that same set of questions/answers with multiple augmented images make complete sense

In [27]:
import os
import sys

In [28]:
# Append the directory to the system path
sys.path.append(os.path.join(os.getcwd(), "pathology-he-autoaugmetation/randaugment/train"))

In [29]:
import pandas as pd
import numpy as np
from randaugment_new_ranges import distort_image_with_randaugment
from PIL import Image
from tqdm import tqdm
import re
import random

In [30]:
random_state = 42
np.random.seed(random_state)
random.seed(random_state)

#### Reading the original data

In [31]:
vqa_data = pd.read_csv('data/path_open_vqa.csv')
vqa_data = vqa_data.rename(columns={vqa_data.columns[2]: 'Image1_Path',
                                    vqa_data.columns[26]: 'Image2_Path',
                                    vqa_data.columns[27]: 'Image3_Path',
                                    vqa_data.columns[28]: 'Image4_Path',
                                    vqa_data.columns[29]: 'Image5_Path',
                                    'Image 1 Magnification ': 'Image1_Mag',
                                    'Image 2 Magnification ': 'Image2_Mag',
                                    'Image 3 Magnification ': 'Image3_Mag',
                                    'Image 4 Magnification ': 'Image4_Mag',
                                    'Image 5 Magnification ': 'Image5_Mag'})

vqa_data = vqa_data.reset_index()
vqa_data['index'] = vqa_data['index'] + 1
vqa_data = vqa_data.rename(columns={'index': 'Case_ID'})
vqa_data.head()

,Case_ID,Timestamp,Pathologist ID,Image1_Path,Organ,Categorization,Regional Anatomy,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Answer 2,...,Open Ended - Wrong Answer 1,Open Ended - Wrong Answer 2,Image2_Mag,Image3_Mag,Image4_Mag,Image5_Mag,Image2_Path,Image3_Path,Image4_Path,Image5_Path
0,1,2/25/2025 13:41:20,CK,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the primary architectural pattern obse...,Nodular pattern,Germinal centers are absent,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2/25/2025 13:46:25,CK,https://drive.google.com/open?id=1igYpj4RL0XKx...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,These would best be characterized as small lym...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2/25/2025 13:51:37,CK,https://drive.google.com/open?id=1DKNZJQJ17SkX...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,This should be considered a highly cellular ti...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2/26/2025 11:30:23,CK,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,Gastrointestinal - Small Intenstine,Infection (Benign),Intestinal villous mucosa,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,Granulomas here are often the consequence of u...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2/26/2025 11:42:26,CK,https://drive.google.com/open?id=1VNr78I5w671g...,Gastrointestinal - Dudenum,Infection (Benign),Intestinal villous mucosa,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Getting only limited columns for the evaluation using only open-ended answers

In [32]:
vqa_data_filt = vqa_data[['Case_ID', 'Pathologist ID', 'Image1_Path', 'Image2_Path', 'Image3_Path', 'Open Ended - Question 1', 'Open Ended - Answer 1', 'Open Ended - Question 2', 'Open Ended - Answer 2']]
vqa_data_filt.head()

,Case_ID,Pathologist ID,Image1_Path,Image2_Path,Image3_Path,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Question 2,Open Ended - Answer 2
0,1,CK,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,NaN,NaN,What is the primary architectural pattern obse...,Nodular pattern,What is a common component of lymph node archi...,Germinal centers are absent
1,2,CK,https://drive.google.com/open?id=1igYpj4RL0XKx...,NaN,NaN,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...
2,3,CK,https://drive.google.com/open?id=1DKNZJQJ17SkX...,NaN,NaN,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,What is the best description for the density a...,This should be considered a highly cellular ti...
3,4,CK,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,NaN,NaN,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,What does the granuloma in the lamina propria ...,Granulomas here are often the consequence of u...
4,5,CK,https://drive.google.com/open?id=1VNr78I5w671g...,NaN,NaN,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,NaN,NaN


Separating the multiple image paths available for the same question to form multiple questions

In [33]:
vqa_data_filt_final = pd.DataFrame()
for index, row in vqa_data_filt.iterrows():
    # Get the first image
    image_path = row['Image1_Path']
    if pd.notnull(image_path):
        row['Image_Path'] = image_path
        row['Image_ID'] = 'img_pathopen_' + str(row['Case_ID']) + '_01'
        vqa_data_filt_final = pd.concat([vqa_data_filt_final, row.to_frame().T], ignore_index=True)
    
    # Get the second image
    image_path = row['Image2_Path']
    if pd.notnull(image_path):
        row['Image_Path'] = image_path
        row['Image_ID'] = 'img_pathopen_' + str(row['Case_ID']) + '_02'
        vqa_data_filt_final = pd.concat([vqa_data_filt_final, row.to_frame().T], ignore_index=True)
    
    # Get the third image
    image_path = row['Image3_Path']
    if pd.notnull(image_path):
        row['Image_Path'] = image_path
        row['Image_ID'] = 'img_pathopen_' + str(row['Case_ID']) + '_03'
        vqa_data_filt_final = pd.concat([vqa_data_filt_final, row.to_frame().T], ignore_index=True)

vqa_data_filt_final.head()

,Case_ID,Pathologist ID,Image1_Path,Image2_Path,Image3_Path,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Question 2,Open Ended - Answer 2,Image_Path,Image_ID
0,1,CK,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,NaN,NaN,What is the primary architectural pattern obse...,Nodular pattern,What is a common component of lymph node archi...,Germinal centers are absent,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,img_pathopen_1_01
1,2,CK,https://drive.google.com/open?id=1igYpj4RL0XKx...,NaN,NaN,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...,https://drive.google.com/open?id=1igYpj4RL0XKx...,img_pathopen_2_01
2,3,CK,https://drive.google.com/open?id=1DKNZJQJ17SkX...,NaN,NaN,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,What is the best description for the density a...,This should be considered a highly cellular ti...,https://drive.google.com/open?id=1DKNZJQJ17SkX...,img_pathopen_3_01
3,4,CK,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,NaN,NaN,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,What does the granuloma in the lamina propria ...,Granulomas here are often the consequence of u...,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,img_pathopen_4_01
4,5,CK,https://drive.google.com/open?id=1VNr78I5w671g...,NaN,NaN,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,NaN,NaN,https://drive.google.com/open?id=1VNr78I5w671g...,img_pathopen_5_01


Removing the un-necessary columns

In [34]:
vqa_data_filt_final = vqa_data_filt_final.drop(columns=['Image1_Path', 'Image2_Path', 'Image3_Path'])
vqa_data_filt_final.head()

,Case_ID,Pathologist ID,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Question 2,Open Ended - Answer 2,Image_Path,Image_ID
0,1,CK,What is the primary architectural pattern obse...,Nodular pattern,What is a common component of lymph node archi...,Germinal centers are absent,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,img_pathopen_1_01
1,2,CK,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...,https://drive.google.com/open?id=1igYpj4RL0XKx...,img_pathopen_2_01
2,3,CK,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,What is the best description for the density a...,This should be considered a highly cellular ti...,https://drive.google.com/open?id=1DKNZJQJ17SkX...,img_pathopen_3_01
3,4,CK,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,What does the granuloma in the lamina propria ...,Granulomas here are often the consequence of u...,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,img_pathopen_4_01
4,5,CK,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,NaN,NaN,https://drive.google.com/open?id=1VNr78I5w671g...,img_pathopen_5_01


### Generating the Google Sheet Containing the mapping between Google Drive Links and Image Names

This will help to map the image names downloaded from collected data with the Google Drive Links we have in our data

In [35]:
# function listFilesInNestedFolders(parentFolderId) {
#   // Get the initial parent folder
#   var parentfolder_id = '1VdFeRx06PrTTCbeMVQ0wQ8ML8QF8sSxM-QWlTcMx7TKQAjpWRuW5-EWJi-DtuBDPeXTbGYNm'
#   const parentFolder = DriveApp.getFolderById(parentfolder_id);
#   var folderlisting = 'File Names and Links - '+ parentfolder_id;

#   var ss = SpreadsheetApp.create(folderlisting);
#   var sheet = ss.getActiveSheet();
#   sheet.appendRow(['name','link']);

#   // Get subfolders in the current folder and process them recursively
#   var subfolders = parentFolder.getFolders();
#   while (subfolders.hasNext()) {
#     var subfolder = subfolders.next();
#     // Get files in the current folder
#     var contents = subfolder.getFiles();
#     var file;
#     var name;
#     var link;
#     while (contents.hasNext()) {
#       file = contents.next();
#       name = file.getName();
#       link = file.getUrl();
#       sheet.appendRow([name,link]);
#     }
#   }
# }

Now Download the file generated which contains mapping between google drive links and names.

In [36]:
data_dir = "data_eval"
file_name = "file_names_and_links_PathOpen_Images.csv"
file_path = os.path.join(data_dir, file_name)
google_drive_name_links = pd.read_csv(file_path)
google_drive_name_links.head()

,name,link
0,ck_HSP_5x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/1SYbssi84BMfkC...
1,ck_BALL_4x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/1XrSu7uRm3s-al...
2,ck_pschwannoma_20x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/1dGZ8MAgdKvT5w...
3,ck_CMV gastritis_4x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/11UMXpBZjiQdo0...
4,ck_rhabdo_5x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/174myggtY1Fj5A...


Extracting the ID's from image links

In [37]:
vqa_data_filt_final['google_drive_image_id'] = vqa_data_filt_final['Image_Path'].apply(lambda x: x.split('=')[-1])
google_drive_name_links['google_drive_image_id'] = google_drive_name_links['link'].apply(lambda x: x.split('/')[-1])

Combining the names of the image files

In [38]:
vqa_data_filt_final = pd.merge(vqa_data_filt_final, google_drive_name_links[['google_drive_image_id', 'name']], on='google_drive_image_id', how='left')
vqa_data_filt_final.head()

,Case_ID,Pathologist ID,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Question 2,Open Ended - Answer 2,Image_Path,Image_ID,google_drive_image_id,name
0,1,CK,What is the primary architectural pattern obse...,Nodular pattern,What is a common component of lymph node archi...,Germinal centers are absent,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,img_pathopen_1_01,1tgv50Q9W4Bm_Mk7my6Ag8mD0JWkQkYvx,MCL_20x - Chandra Krishnan.jpg
1,2,CK,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...,https://drive.google.com/open?id=1igYpj4RL0XKx...,img_pathopen_2_01,1igYpj4RL0XKxq0u5qlnv6ZvqJfFjGO4u,MCL_200x - Chandra Krishnan.jpg
2,3,CK,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,What is the best description for the density a...,This should be considered a highly cellular ti...,https://drive.google.com/open?id=1DKNZJQJ17SkX...,img_pathopen_3_01,1DKNZJQJ17SkXUrrO7jHMiKD8J6ittzus,MCL_400x - Chandra Krishnan.jpg
3,4,CK,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,What does the granuloma in the lamina propria ...,Granulomas here are often the consequence of u...,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,img_pathopen_4_01,1jUe0z6wlZ0s9PZUgVj1yCL7UZav23jwz,Duodenum granuloma_40x - Chandra Krishnan.jpg
4,5,CK,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,NaN,NaN,https://drive.google.com/open?id=1VNr78I5w671g...,img_pathopen_5_01,1VNr78I5w671gkrIyHs7AC75yrjb1lU6_,Duodenum granuloma_100x - Chandra Krishnan.jpg


Creating a dictionary between Image_ID and name

In [39]:
name_imageid_dict = dict(zip(vqa_data_filt_final['name'], vqa_data_filt_final['Image_ID']))

### Reading the dataset files

Preparing the augmented images for each filename

In [40]:
source_dataset_dir = "PathOPEN_Eval_Images"
destination_dataset_dir = "PathOPEN_Eval_Images_Augmented"
pathopen_file_names = os.listdir(source_dataset_dir)

Shuffle the list and taking 50 images

In [41]:
random.seed(random_state)
random.shuffle(pathopen_file_names)
pathopen_file_names

['ck_molluscum_40x - Chandra Krishnan.jpg',
 'ck_fnh_5x - Chandra Krishnan.jpeg',
 'ac_img29_100x - Amy Coffey.jpeg',
 'ck_clear cell carcinoma_20x - Chandra Krishnan.jpeg',
 'ac_img47_200x - Amy Coffey.jpeg',
 'ac_img45_100x - Amy Coffey.jpeg',
 'ck_invasive lobular_20x - Chandra Krishnan.jpeg',
 'ck_cellular fibroadenoma_20x - Chandra Krishnan.jpg',
 'ac_img32_200x - Amy Coffey.jpeg',
 'ck_liverhepc_20x - Chandra Krishnan.jpeg',
 'ck_invasive lobular_10x - Chandra Krishnan.jpeg',
 'ck_lch_10x - Chandra Krishnan.jpeg',
 'ac_img51_40x - Amy Coffey.jpeg',
 'ac_img1_400x - Amy Coffey.jpg',
 'ck_colon carcinoma_100x - Chandra Krishnan.jpg',
 'ck_streak ovary_2x - Chandra Krishnan.jpg',
 'MCL_200x - Chandra Krishnan.jpg',
 'ck_ganglioneuroma_20x - Chandra Krishnan.jpeg',
 'ck_corpus luteum_20x - Chandra Krishnan.jpg',
 'ck_plexiform schwannoma_100x - Chandra Krishnan.jpg',
 'ac_img5_100x - Amy Coffey.jpg',
 'ck_colon carcinoma_200x - Chandra Krishnan.jpg',
 'ck_inflamed thyroid_10x - Chand

In [42]:
pathopen_file_names_top50 = pathopen_file_names[:50]

Now peform the augmentation on these images

- Keep the code uncommented if not to be used since it takes along time to execute

In [43]:
# TOTAL_AUGMENTAIONS = 3  # Number of augmentations per image
# for src_image in pathopen_file_names_top50:
#     src_image_path = os.path.join(source_dataset_dir, src_image)
#     src_image_id = name_imageid_dict[src_image]
#     dst_image_aug_folder = os.path.join(destination_dataset_dir, src_image_id + "_augmented")
#     os.makedirs(dst_image_aug_folder, exist_ok=True)

#     pil_image = Image.open(src_image_path).convert("RGB")
#     np_array_img = np.array(pil_image)
#     for i in tqdm(range(TOTAL_AUGMENTAIONS), desc=f"Augmenting {src_image_id}"):
#         # Apply RandAugment with specified parameters
#         conv_img_arr = distort_image_with_randaugment(np_array_img, num_layers=5, magnitude=5, randomize=True, randaugment_transforms_set='custom')
#         conv_img = Image.fromarray(conv_img_arr)
#         conv_img.save(os.path.join(dst_image_aug_folder, src_image_id + f"_aug_{i}.png"))

### Source the data onto the Google Drive

Now upload these images onto the Google Drive. Then run a Google App Script to get the relation between names and Google Drive Links using the following script

In [44]:
# function listFilesInNestedFolders(parentFolderId) {
#   // Get the initial parent folder
#   var parentfolder_id = '1jYUpk4fDC89VAp-bsBoZZfCjARUyway4'
#   const parentFolder = DriveApp.getFolderById(parentfolder_id);
#   var folderlisting = 'File Names and Links - '+ parentfolder_id;

#   var ss = SpreadsheetApp.create(folderlisting);
#   var sheet = ss.getActiveSheet();
#   sheet.appendRow(['name','link']);

#   // Get subfolders in the current folder and process them recursively
#   var subfolders = parentFolder.getFolders();
#   while (subfolders.hasNext()) {
#     var subfolder = subfolders.next();
#     // Get files in the current folder
#     var contents = subfolder.getFiles();
#     var file;
#     var name;
#     var link;
#     while (contents.hasNext()) {
#       file = contents.next();
#       name = file.getName();
#       link = file.getUrl();
#       sheet.appendRow([name,link]);
#     }
#   }
# }

Now put the generated excel sheet into `data_eval` so that we can map the image_ids with the google drive links

In [45]:
data_eval_dir = "data_eval"
file_name_augmented = "file_names_and_links_PathOpen_Augmented_Images.csv"
file_path_augmented = os.path.join(data_eval_dir, file_name_augmented)
google_drive_name_links_augmented = pd.read_csv(file_path_augmented)
google_drive_name_links_augmented.head()

,name,link
0,img_pathopen_156_03_aug_2.png,https://drive.google.com/file/d/111GjvR7s8LxmP...
1,img_pathopen_156_03_aug_1.png,https://drive.google.com/file/d/1EsQTKf2IXpHrW...
2,img_pathopen_156_03_aug_0.png,https://drive.google.com/file/d/1uiIOhr6rY8Y0k...
3,img_pathopen_154_01_aug_2.png,https://drive.google.com/file/d/1LrOWitcO_OrVN...
4,img_pathopen_154_01_aug_1.png,https://drive.google.com/file/d/13UYuNjJbjd0Rd...


Extracting the image id from name

In [46]:
google_drive_name_links_augmented['Image_ID'] = google_drive_name_links_augmented['name'].apply(lambda x: re.match(r'(img_pathopen_\d+_\d+)(.*)', x).group(1))
google_drive_name_links_augmented = google_drive_name_links_augmented.rename(columns={'name': 'Augmented_Image_ID', 'link': 'Augmented_Image_Link'})
google_drive_name_links_augmented.head()

,Augmented_Image_ID,Augmented_Image_Link,Image_ID
0,img_pathopen_156_03_aug_2.png,https://drive.google.com/file/d/111GjvR7s8LxmP...,img_pathopen_156_03
1,img_pathopen_156_03_aug_1.png,https://drive.google.com/file/d/1EsQTKf2IXpHrW...,img_pathopen_156_03
2,img_pathopen_156_03_aug_0.png,https://drive.google.com/file/d/1uiIOhr6rY8Y0k...,img_pathopen_156_03
3,img_pathopen_154_01_aug_2.png,https://drive.google.com/file/d/1LrOWitcO_OrVN...,img_pathopen_154_01
4,img_pathopen_154_01_aug_1.png,https://drive.google.com/file/d/13UYuNjJbjd0Rd...,img_pathopen_154_01


### Creating the final dataset



Joining the filtered dataset on `Image_Id` to get the open-ended questions/answers for each augmented image since each augmented image should have the same questions/answers as the parent image for a fair evaluation

In [47]:
vqa_data_augmented_images = pd.merge(vqa_data_filt_final, google_drive_name_links_augmented, on='Image_ID', how='inner')
vqa_data_augmented_images.head()

,Case_ID,Pathologist ID,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Question 2,Open Ended - Answer 2,Image_Path,Image_ID,google_drive_image_id,name,Augmented_Image_ID,Augmented_Image_Link
0,2,CK,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...,https://drive.google.com/open?id=1igYpj4RL0XKx...,img_pathopen_2_01,1igYpj4RL0XKxq0u5qlnv6ZvqJfFjGO4u,MCL_200x - Chandra Krishnan.jpg,img_pathopen_2_01_aug_2.png,https://drive.google.com/file/d/1S-FsI5LzL57kQ...
1,2,CK,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...,https://drive.google.com/open?id=1igYpj4RL0XKx...,img_pathopen_2_01,1igYpj4RL0XKxq0u5qlnv6ZvqJfFjGO4u,MCL_200x - Chandra Krishnan.jpg,img_pathopen_2_01_aug_1.png,https://drive.google.com/file/d/1sT40Ed8p6d6qf...
2,2,CK,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...,https://drive.google.com/open?id=1igYpj4RL0XKx...,img_pathopen_2_01,1igYpj4RL0XKxq0u5qlnv6ZvqJfFjGO4u,MCL_200x - Chandra Krishnan.jpg,img_pathopen_2_01_aug_0.png,https://drive.google.com/file/d/1DM6H3HhdtBCFk...
3,18,CK,What is the best way to describe the changes s...,The lamina propria in this biopsy shows fibros...,The fragments of tissue on the left edge of th...,The left edge shows sampling of surface mucosa...,https://drive.google.com/open?id=1WavtKmBC8Epj...,img_pathopen_18_01,1WavtKmBC8EpjnJ3rZOZ4lvlbT9w4eQQu,ck_colon carcinoma_100x - Chandra Krishnan.jpg,img_pathopen_18_01_aug_2.png,https://drive.google.com/file/d/12vAnVKBLJb5El...
4,18,CK,What is the best way to describe the changes s...,The lamina propria in this biopsy shows fibros...,The fragments of tissue on the left edge of th...,The left edge shows sampling of surface mucosa...,https://drive.google.com/open?id=1WavtKmBC8Epj...,img_pathopen_18_01,1WavtKmBC8EpjnJ3rZOZ4lvlbT9w4eQQu,ck_colon carcinoma_100x - Chandra Krishnan.jpg,img_pathopen_18_01_aug_1.png,https://drive.google.com/file/d/1OItNxb9tOgdLU...


Creating the final csv file to be uploaded for evaluation

In [48]:
vqa_data_augmented_images_clean = vqa_data_augmented_images[['Case_ID', 'Augmented_Image_ID', 'Augmented_Image_Link', 'Open Ended - Question 1', 'Open Ended - Answer 1', 'Open Ended - Question 2', 'Open Ended - Answer 2']]
vqa_data_augmented_images_clean

,Case_ID,Augmented_Image_ID,Augmented_Image_Link,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Question 2,Open Ended - Answer 2
0,2,img_pathopen_2_01_aug_2.png,https://drive.google.com/file/d/1S-FsI5LzL57kQ...,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...
1,2,img_pathopen_2_01_aug_1.png,https://drive.google.com/file/d/1sT40Ed8p6d6qf...,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...
2,2,img_pathopen_2_01_aug_0.png,https://drive.google.com/file/d/1DM6H3HhdtBCFk...,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,What is the best description for the cell size...,These would best be characterized as small lym...
3,18,img_pathopen_18_01_aug_2.png,https://drive.google.com/file/d/12vAnVKBLJb5El...,What is the best way to describe the changes s...,The lamina propria in this biopsy shows fibros...,The fragments of tissue on the left edge of th...,The left edge shows sampling of surface mucosa...
4,18,img_pathopen_18_01_aug_1.png,https://drive.google.com/file/d/1OItNxb9tOgdLU...,What is the best way to describe the changes s...,The lamina propria in this biopsy shows fibros...,The fragments of tissue on the left edge of th...,The left edge shows sampling of surface mucosa...
...,...,...,...,...,...,...,...
169,154,img_pathopen_154_01_aug_1.png,https://drive.google.com/file/d/13UYuNjJbjd0Rd...,What is the infectious disease pathology depic...,The image shows gastric antral mucosa with chr...,What is the anatomic location of this biopsy t...,The biopsy represents sampling of gastric muco...
170,154,img_pathopen_154_01_aug_0.png,https://drive.google.com/file/d/1p1kP5QV8xlofz...,What is the infectious disease pathology depic...,The image shows gastric antral mucosa with chr...,What is the anatomic location of this biopsy t...,The biopsy represents sampling of gastric muco...
171,156,img_pathopen_156_03_aug_2.png,https://drive.google.com/file/d/111GjvR7s8LxmP...,Does the image representing sampling of the bo...,"No, this image shows sampling of the medullary...",What are the diagnostic possibilities that can...,The image shows sampling of the medullary marr...
172,156,img_pathopen_156_03_aug_1.png,https://drive.google.com/file/d/1EsQTKf2IXpHrW...,Does the image representing sampling of the bo...,"No, this image shows sampling of the medullary...",What are the diagnostic possibilities that can...,The image shows sampling of the medullary marr...


### Suffling the dataframe with fixed seed

In [49]:
vqa_data_augmented_images_clean_shuffle = vqa_data_augmented_images_clean.sample(frac=1, random_state=random_state).reset_index(drop=True)
vqa_data_augmented_images_clean_shuffle.head()

,Case_ID,Augmented_Image_ID,Augmented_Image_Link,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Question 2,Open Ended - Answer 2
0,148,img_pathopen_148_01_aug_0.png,https://drive.google.com/file/d/1iwqNhkvNBL4iI...,What are the main inflammatory cell types seen...,The biopsy shows a cellular lesion comprised o...,What is the most likely diagnosis suggested by...,The lesion shows large atypical mononuclear ce...
1,142,img_pathopen_142_03_aug_2.png,https://drive.google.com/file/d/1en7HkPZzhxKnb...,Based on the image depicting a section of the ...,The image shows typical changes of Graves dise...,What is the best histologic description of thi...,The image shows typical changes of Grave's dis...
2,117,img_pathopen_117_01_aug_1.png,https://drive.google.com/file/d/1yiLkYnwp4-e4e...,What is the classic description of the tumor d...,The image depicts a wilms tumor showing classi...,In this image depicting a sample from a kidney...,"This is a wilms tumor and the blastema, primit..."
3,129,img_pathopen_129_02_aug_1.png,https://drive.google.com/file/d/1oi6xVKJa5xHxb...,What is the general category of lesion and are...,The lesion generically shows a fibroepithelial...,What are the histologic findings in this image...,This fibroepithelial lesion is likely not to r...
4,137,img_pathopen_137_03_aug_0.png,https://drive.google.com/file/d/1mbMRpC_Bywi_f...,What is the abnormality observed in this image...,The image depicts a molar pregnancy with unifo...,What is the expected chromosomal findings pred...,The image depicts likely complete hydatidiform...


Dividing the dataframe into five small dfs

In [50]:
df_part_len = int(len(vqa_data_augmented_images_clean_shuffle)/5)
vqa_data_augmented_images_clean_shuffle_part1 = vqa_data_augmented_images_clean_shuffle.iloc[0:df_part_len]
vqa_data_augmented_images_clean_shuffle_part2 = vqa_data_augmented_images_clean_shuffle.iloc[df_part_len:2*df_part_len]
vqa_data_augmented_images_clean_shuffle_part3 = vqa_data_augmented_images_clean_shuffle.iloc[2*df_part_len:3*df_part_len]
vqa_data_augmented_images_clean_shuffle_part4 = vqa_data_augmented_images_clean_shuffle.iloc[3*df_part_len:4*df_part_len]
vqa_data_augmented_images_clean_shuffle_part5 = vqa_data_augmented_images_clean_shuffle.iloc[4*df_part_len:len(vqa_data_augmented_images_clean_shuffle)]

In [51]:
vqa_data_augmented_images_clean_shuffle_part1.to_csv('data_eval/pathopen_vqa_augmented_part1.csv', index=False)
vqa_data_augmented_images_clean_shuffle_part2.to_csv('data_eval/pathopen_vqa_augmented_part2.csv', index=False)
vqa_data_augmented_images_clean_shuffle_part3.to_csv('data_eval/pathopen_vqa_augmented_part3.csv', index=False)
vqa_data_augmented_images_clean_shuffle_part4.to_csv('data_eval/pathopen_vqa_augmented_part4.csv', index=False)
vqa_data_augmented_images_clean_shuffle_part5.to_csv('data_eval/pathopen_vqa_augmented_part5.csv', index=False)